In [12]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [13]:
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [14]:
from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings(model="text-embedding-3-small")
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x14fa8a470>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x14fa890f0>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [15]:
## Single text embeddings
single_text="Langchain and Rag are amazing frameworks and projects to work on"
single_embedding = embeddings.embed_query(single_text)
print(f"length of embedding is {len(single_embedding)}")
print(f"Sample embedding at index 1535 is {single_embedding[1535]}")
print(f"Overall embedding are {single_embedding}")

length of embedding is 1536
Sample embedding at index 1535 is -0.0051249428652226925
Overall embedding are [-0.05006517842411995, -0.031117182224988937, -0.0034983009099960327, -0.0032226566690951586, 0.03250561282038689, -0.03166166692972183, -0.014728255569934845, 0.0014148041373118758, -0.011046191677451134, -0.034193508327007294, 0.01806321181356907, 0.005101121496409178, -0.03849492222070694, 0.05003795400261879, 0.005791934207081795, 0.014496849849820137, -0.00023140512348618358, -0.06239771097898483, 0.0406184047460556, 0.06512012332677841, -0.0015219991328194737, -0.006230923347175121, -0.01804959960281849, 0.0329684242606163, -0.0069897957146167755, -0.008092373609542847, -0.007663593161851168, 0.06310553848743439, 0.017627624794840813, -0.019723882898688316, 0.00024927096092142165, -0.033921267837285995, -0.00677540572360158, 0.033730700612068176, 0.015163841657340527, 0.022078771144151688, -0.005808949004858732, 0.0037841543089598417, -0.008024312555789948, 0.022664090618491

### Multiple texts at once and generating embeddings

In [16]:
# Example 2: Multiple texts at once
multiple_texts = [
    "Python is a programming language",
    "LangChain is a framework for LLM applications",
    "Embeddings convert text to numbers",
    "Vectors can be compared for similarity"
]

In [19]:
multiple_embeddings= embeddings.embed_documents(multiple_texts)

for i in range(len(multiple_texts)):
    print(f" Embedding dimension 1500 of sentence {i} is {multiple_embeddings[i][1500]}")

 Embedding dimension 1500 of sentence 0 is 0.009836210869252682
 Embedding dimension 1500 of sentence 1 is -0.00687266094610095
 Embedding dimension 1500 of sentence 2 is -0.006322159897536039
 Embedding dimension 1500 of sentence 3 is -0.0002819180372171104


### Cosine similarity with OpenAI Embeddings

In [36]:
sentences = [
    "The cat sat on the mat",
    "The black cat is eating food",
    "A feline rested on the rug",
    "The dog played in the yard",
    "I love programming in Python",
    "Python is my favorite programming language"
]

In [37]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


In [41]:
import numpy as np
def cosine_similarity(vec1,vec2):
    dot_product = np.dot(vec1,vec2)
    norm_1 = np.linalg.norm(vec1)
    norm_2 = np.linalg.norm(vec2)
    similarity= dot_product/(norm_1)*(norm_2)
    if similarity > 0.6:
        print(f"strong similarity (cosine similarity value {similarity})")
    else:
        print(f"Weak similarity (cosine similarity value {similarity})")
    return similarity
    

In [39]:
sentence_embedding = embeddings.embed_documents(sentences)


In [42]:
for i in range(len(sentences)):
    for j in range(i+1,len(sentences)):
        similarity=cosine_similarity(sentence_embedding[i],sentence_embedding[j])
        
        print(f"'{sentences[i]}' vs '{sentences[j]}'")
        print(f"Similarity: {similarity:.3f}\n")
        

Weak similarity (cosine similarity value 0.4530527167379754)
'The cat sat on the mat' vs 'The black cat is eating food'
Similarity: 0.453

strong similarity (cosine similarity value 0.655520266667793)
'The cat sat on the mat' vs 'A feline rested on the rug'
Similarity: 0.656

Weak similarity (cosine similarity value 0.3243763523269939)
'The cat sat on the mat' vs 'The dog played in the yard'
Similarity: 0.324

Weak similarity (cosine similarity value 0.08954862171349287)
'The cat sat on the mat' vs 'I love programming in Python'
Similarity: 0.090

Weak similarity (cosine similarity value 0.12017014716475227)
'The cat sat on the mat' vs 'Python is my favorite programming language'
Similarity: 0.120

Weak similarity (cosine similarity value 0.495332482013278)
'The black cat is eating food' vs 'A feline rested on the rug'
Similarity: 0.495

Weak similarity (cosine similarity value 0.2380587424170048)
'The black cat is eating food' vs 'The dog played in the yard'
Similarity: 0.238

Weak si

### Scemantic Search

In [45]:
### Example- Semantic Search- Retireve the similar sentence
# Test semantic search
documents = [
    "LangChain is a framework for developing applications powered by language models",
    "Python is a high-level programming language",
    "Machine learning is a subset of artificial intelligence",
    "Embeddings convert text into numerical vectors",
    "The weather today is sunny and warm"
]
query="What is Langchain?"

In [46]:
def scemantic_search(query,documents,embedding_models,top_k=3):
    """Simple scemantic search implementation"""
    
    query_embedding = embedding_models.embed_query(query)
    doc_embedding = embedding_models.embed_documents(documents)
    similarities =[]
    
    for i,doc_embed in enumerate(doc_embedding):
        similarity = cosine_similarity(query_embedding,doc_embed)
        similarities.append((similarity,documents[i]))
    similarities.sort(reverse=True)
    return similarities[:top_k]

    
    
        

In [47]:
results = scemantic_search(query,documents,embeddings)
results

strong similarity (cosine similarity value 0.6756028654080636)
Weak similarity (cosine similarity value 0.13031421097970233)
Weak similarity (cosine similarity value 0.06433764200000079)
Weak similarity (cosine similarity value 0.10107533297211645)
Weak similarity (cosine similarity value 0.002155653771581921)


[(np.float64(0.6756028654080636),
  'LangChain is a framework for developing applications powered by language models'),
 (np.float64(0.13031421097970233),
  'Python is a high-level programming language'),
 (np.float64(0.10107533297211645),
  'Embeddings convert text into numerical vectors')]